# Part 2, step 3: embed-and-pool (Liam)

Run from `part2/`. Embeddings: `python -m part2.prepare_embeddings` from the repo root (once).

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT))
from part2 import embcv, wordvecs as WV  # noqa: E402
from shared import analysis as A, eval_harness as H  # noqa: E402

# Cesare's step 2.5 probe: logistic regression on mean-pooled vectors, 50 shared folds
R = H.load_results("emb-*__logreg.json")
A.results_table(R)

,folds,minutes,balanced_accuracy,accuracy,macro_f1
system,,,,,
emb-ft-crawl-mean-promptcentred/logreg,50,0.0,0.4060 ± 0.0245,0.4061 ± 0.0245,0.3949 ± 0.0263
emb-ft-wiki-mean-promptcentred/logreg,50,0.0,0.3994 ± 0.0177,0.3995 ± 0.0177,0.3864 ± 0.0191
emb-glove-840b-mean-promptcentred/logreg,50,0.0,0.3868 ± 0.0225,0.3869 ± 0.0225,0.3770 ± 0.0229
emb-w2v-gnews-mean-promptcentred/logreg,50,0.0,0.3462 ± 0.0197,0.3463 ± 0.0196,0.3367 ± 0.0212
emb-ft-wiki-mean/logreg,50,0.0,0.2960 ± 0.0237,0.2961 ± 0.0237,0.2800 ± 0.0244
emb-ft-crawl-mean/logreg,50,0.1,0.2853 ± 0.0214,0.2854 ± 0.0214,0.2713 ± 0.0215
emb-glove-840b-mean/logreg,50,0.0,0.2698 ± 0.0239,0.2698 ± 0.0239,0.2564 ± 0.0245
emb-w2v-gnews-mean/logreg,50,0.1,0.2466 ± 0.0242,0.2467 ± 0.0242,0.2331 ± 0.0236


In [10]:
# Baseline: tutorial embed-and-pool + MLP head (50x50, 20 epochs, frozen ft-crawl), 5 folds
tok, emb = WV.load_subset(WV.CHOSEN)
mlp = embcv.run_cv_torch(tok, emb, "embed-and-pool", name="ft-crawl-pool/mlp-50x50", verbose=False)
H.save_result(mlp)
A.comparison_table({**R, mlp["name"]: mlp}, pairs=[("emb-ft-crawl-mean/logreg", mlp["name"])])[0]

  0%|          | 0/20 [00:00<?, ?it/s]

  0%|          | 0/20 [00:00<?, ?it/s]

  0%|          | 0/20 [00:00<?, ?it/s]

  0%|          | 0/20 [00:00<?, ?it/s]

  0%|          | 0/20 [00:00<?, ?it/s]

ft-crawl-pool/mlp-50x50      bal.acc 0.1981 ± 0.0272  (5 folds, 32s)


,A,B,folds,mean B−A,P(A>B),P(ROPE),P(B>A)
0,emb-ft-crawl-mean/logreg,ft-crawl-pool/mlp-50x50,5,-0.0972,0.997,0.0016,0.0014


In [34]:
# More Epochs: tutorial embed-and-pool + MLP head (50x50, 100 epochs, frozen ft-crawl), 5 folds
tok, emb = WV.load_subset(WV.CHOSEN)
mlp = embcv.run_cv_torch(tok, emb, "embed-and-pool", name="ft-crawl-pool/mlp-50x50-e100", verbose=False, opt_kwargs={"learning_rate": 1e-3, "weight_decay": 1e-4, "batch_size": 64, "num_epochs": 100})
H.save_result(mlp)
A.comparison_table({**R, mlp["name"]: mlp}, pairs=[("emb-ft-crawl-mean/logreg", mlp["name"])])[0]

  0%|          | 0/100 [00:00<?, ?it/s]

  0%|          | 0/100 [00:00<?, ?it/s]

  0%|          | 0/100 [00:00<?, ?it/s]

  0%|          | 0/100 [00:00<?, ?it/s]

  0%|          | 0/100 [00:00<?, ?it/s]

ft-crawl-pool/mlp-50x50-e100 bal.acc 0.2306 ± 0.0233  (5 folds, 89s)


,A,B,folds,mean B−A,P(A>B),P(ROPE),P(B>A)
0,emb-ft-crawl-mean/logreg,ft-crawl-pool/mlp-50x50-e100,5,-0.0648,0.9866,0.0087,0.0048


In [14]:
# Checking which epoch was the best
import pandas as pd
pd.DataFrame(mlp["folds"])[["fold", "best_epoch", "balanced_accuracy"]]

,fold,best_epoch,balanced_accuracy
0,0,65,0.220109
1,1,56,0.273148
2,2,43,0.233796
3,3,24,0.203704
4,4,49,0.222222


MLP is fully trained. More epochs would not help. So next trying it with prompt centering.

In [15]:
from shared.data import load_data
from shared.prompt_groups import prompt_centre

D = load_data()
X = WV.pool(tok, D.train["text"].tolist())             # mean-pooled ft-crawl vectors (= the encoder output)
folds = H.load_folds(D.train["id"].tolist(), repeats=1)  # same 5 folds as the embcv runs
grid = {"hidden": [50], "dropout": [0.1], "lr": [1e-3], "weight_decay": [1e-4]}  # one fixed setting, no tuning

raw = H.run_cv(X, D.y, D.groups, folds, "mlp", grid=grid, name="ft-crawl-mean/mlp-50")
cen = H.run_cv(prompt_centre(X, D.groups), D.y, D.groups, folds, "mlp", grid=grid,
               name="ft-crawl-mean-promptcentred/mlp-50")
H.save_result(raw); H.save_result(cen)

A.comparison_table({**R, raw["name"]: raw, cen["name"]: cen},
                   pairs=[(raw["name"], cen["name"]),                                   # centring effect, same MLP
                          ("emb-ft-crawl-mean-promptcentred/logreg", cen["name"])])[0]  # vs centred logreg

ft-crawl-mean/mlp-50         bal.acc 0.2736 ± 0.0133  (5 folds, 15s)
ft-crawl-mean-promptcentred/mlp-50 bal.acc 0.4079 ± 0.0167  (5 folds, 13s)


,A,B,folds,mean B−A,P(A>B),P(ROPE),P(B>A)
0,ft-crawl-mean/mlp-50,ft-crawl-mean-promptcentred/mlp-50,5,0.1343,0.0003,0.0002,0.9994
1,emb-ft-crawl-mean-promptcentred/logreg,ft-crawl-mean-promptcentred/mlp-50,5,0.0009,0.1869,0.5885,0.2247


After prompt centering, MLP and logreg are undefined. So next I try MLP on all 50 folds to compare.

In [17]:
folds50 = H.load_folds(D.train["id"].tolist())   # all 10 repeats = 50 folds
cen50 = H.run_cv(prompt_centre(X, D.groups), D.y, D.groups, folds50, "mlp", grid=grid,
                 name="ft-crawl-mean-promptcentred/mlp-50")

H.save_result(cen50)   # same name, so it replaces the 5-fold result in shared/results/

A.comparison_table({**R, raw["name"]: raw, cen50["name"]: cen50},
                   pairs=[(raw["name"], cen50["name"]),                                   # centring effect, same MLP
                          ("emb-ft-crawl-mean-promptcentred/logreg", cen50["name"])])[0]  # vs centred logreg

ft-crawl-mean-promptcentred/mlp-50 bal.acc 0.4024 ± 0.0221  (50 folds, 33s)


,A,B,folds,mean B−A,P(A>B),P(ROPE),P(B>A)
0,ft-crawl-mean/mlp-50,ft-crawl-mean-promptcentred/mlp-50,5,0.1343,0.0003,0.0002,0.9994
1,emb-ft-crawl-mean-promptcentred/logreg,ft-crawl-mean-promptcentred/mlp-50,50,-0.0035,0.2474,0.6750,0.0776


With all 50 folds it is still undecided between logreg and mlp. So I try out different MLP settings.

In [18]:
Xc = prompt_centre(X, D.groups)
tuned = H.run_cv(Xc, D.y, D.groups, folds, "mlp", name="ft-crawl-mean-promptcentred/mlp-tuned")
deep  = H.run_cv(Xc, D.y, D.groups, folds, "resmlp", name="ft-crawl-mean-promptcentred/resmlp-tuned")
H.save_result(tuned); H.save_result(deep)

new = {r["name"]: r for r in [tuned, deep]}
table, cmps = A.comparison_table({**R, **new},
                                 pairs=[("emb-ft-crawl-mean-promptcentred/logreg", n) for n in new])
table

ft-crawl-mean-promptcentred/mlp-tuned bal.acc 0.4144 ± 0.0175  (5 folds, 76s)
ft-crawl-mean-promptcentred/resmlp-tuned bal.acc 0.3927 ± 0.0177  (5 folds, 204s)


,A,B,folds,mean B−A,P(A>B),P(ROPE),P(B>A)
0,emb-ft-crawl-mean-promptcentred/logreg,ft-crawl-mean-promptcentred/mlp-tuned,5,0.0073,0.1843,0.3734,0.4422
1,emb-ft-crawl-mean-promptcentred/logreg,ft-crawl-mean-promptcentred/resmlp-tuned,5,-0.0143,0.5958,0.2960,0.1082


In [25]:
# Finding out which setting performed best:
f = pd.DataFrame(tuned["folds"])
pd.set_option("display.max_colwidth", None)   # never shorten cell text
pd.set_option("display.width", 250)           # allow wider rows before wrapping
display(f[["fold", "params", "epochs", "inner_val", "balanced_accuracy"]])

f["params"].astype(str).value_counts()

,fold,params,epochs,inner_val,balanced_accuracy
0,0,"{'hidden': 256, 'dropout': 0.5, 'lr': 0.001, 'weight_decay': 0.0001}",22,0.458333,0.418981
1,1,"{'hidden': 256, 'dropout': 0.2, 'lr': 0.001, 'weight_decay': 0.0001}",9,0.391667,0.416667
2,2,"{'hidden': 256, 'dropout': 0.2, 'lr': 0.0003, 'weight_decay': 0.0001}",23,0.411111,0.435185
3,3,"{'hidden': 256, 'dropout': 0.5, 'lr': 0.0003, 'weight_decay': 0.01}",40,0.413889,0.418981
4,4,"{'hidden': 1024, 'dropout': 0.5, 'lr': 0.0003, 'weight_decay': 0.0001}",16,0.433333,0.381944


params
{'hidden': 256, 'dropout': 0.5, 'lr': 0.001, 'weight_decay': 0.0001}      1
{'hidden': 256, 'dropout': 0.2, 'lr': 0.001, 'weight_decay': 0.0001}      1
{'hidden': 256, 'dropout': 0.2, 'lr': 0.0003, 'weight_decay': 0.0001}     1
{'hidden': 256, 'dropout': 0.5, 'lr': 0.0003, 'weight_decay': 0.01}       1
{'hidden': 1024, 'dropout': 0.5, 'lr': 0.0003, 'weight_decay': 0.0001}    1
Name: count, dtype: int64

In [37]:
# Additionally give the Head better Input. Here I test “function words + punctuation” average, what Cesare already tested, which scored as well as averaging all tokens.
import re, nltk
import numpy as np
STOP = set(nltk.corpus.stopwords.words("english")) | {"'s", "'t", "'re", "'ve", "'ll", "'d", "'m", "s", "t"}
is_function = lambda s: s.lower() in STOP or not re.search(r"[^\W\d_]", s)  # stopword, punctuation or number

texts = D.train["text"].tolist()
X2 = np.hstack([X, WV.pool(tok, texts, keep=is_function)])                 # all tokens | function words + punct
two = H.run_cv(prompt_centre(X2, D.groups), D.y, D.groups, folds50, "mlp",
               name="ft-crawl-mean+func-promptcentred/mlp-nested")
H.save_result(two)

ft-crawl-mean+func-promptcentred/mlp-nested bal.acc 0.4359 ± 0.0222  (50 folds, 999s)


WindowsPath('c:/Users/Liam/NLP Labs/nlp1-team/shared/results/ft-crawl-mean+func-promptcentred__mlp-nested.json')

In [38]:
new = {r["name"]: r for r in [tuned, deep, two]}
A.comparison_table({**R, **new}, pairs=[("emb-ft-crawl-mean-promptcentred/logreg", n) for n in new])[0]

,A,B,folds,mean B−A,P(A>B),P(ROPE),P(B>A)
0,emb-ft-crawl-mean-promptcentred/logreg,ft-crawl-mean-promptcentred/mlp-tuned,5,0.0073,0.1843,0.3734,0.4422
1,emb-ft-crawl-mean-promptcentred/logreg,ft-crawl-mean-promptcentred/resmlp-tuned,5,-0.0143,0.5958,0.2960,0.1082
2,emb-ft-crawl-mean-promptcentred/logreg,ft-crawl-mean+func-promptcentred/mlp-nested,50,0.0299,0.0004,0.0401,0.9595


Looks like MLP is now decided better than logreg, but comparison is unfair so far, since the logreg was not running with added function word cahnnel.

In [39]:
two_lr = H.run_cv(prompt_centre(X2, D.groups), D.y, D.groups, folds50, "logreg",
                  name="emb-ft-crawl-mean+func-promptcentred/logreg")
H.save_result(two_lr)
A.comparison_table({**R, two["name"]: two, two_lr["name"]: two_lr},
                   pairs=[("emb-ft-crawl-mean-promptcentred/logreg", two_lr["name"]),   # input effect
                          (two_lr["name"], two["name"])])[0]                           # head effect, same input

emb-ft-crawl-mean+func-promptcentred/logreg bal.acc 0.4304 ± 0.0190  (50 folds, 19s)


,A,B,folds,mean B−A,P(A>B),P(ROPE),P(B>A)
0,emb-ft-crawl-mean-promptcentred/logreg,emb-ft-crawl-mean+func-promptcentred/logreg,50,0.0244,0.0002,0.0594,0.9404
1,emb-ft-crawl-mean+func-promptcentred/logreg,ft-crawl-mean+func-promptcentred/mlp-nested,50,0.0055,0.0429,0.6503,0.3068


So overall it still remains, that logreg and mlp are undecided, which is why we choose the simple model (logreg), and try out different embedding sizes.

In [31]:
V = np.asarray(emb.vectors[2:], np.float64); mu = V.mean(0)
_, _, Vt = np.linalg.svd(V - mu, full_matrices=False)   # same projection as emb.pca(k)
for k in (50, 100, 200):
    Xk = (X2[:, :300] - mu) @ Vt[:k].T, (X2[:, 300:] - mu) @ Vt[:k].T   # both channels
    r = H.run_cv(prompt_centre(np.hstack(Xk), D.groups), D.y, D.groups, folds50, "logreg",
                 name=f"emb-ft-crawl-pca{k}-mean+func-promptcentred/logreg")
    H.save_result(r)

emb-ft-crawl-pca50-mean+func-promptcentred/logreg bal.acc 0.3697 ± 0.0184  (50 folds, 18s)
emb-ft-crawl-pca100-mean+func-promptcentred/logreg bal.acc 0.3741 ± 0.0208  (50 folds, 9s)
emb-ft-crawl-pca200-mean+func-promptcentred/logreg bal.acc 0.3856 ± 0.0194  (50 folds, 16s)


In [32]:
base = "emb-ft-crawl-mean+func-promptcentred/logreg"
P = H.load_results("emb-ft-crawl-*mean+func-promptcentred__logreg.json")
names = [f"emb-ft-crawl-pca{k}-mean+func-promptcentred/logreg" for k in (50, 100, 200)]

display(A.results_table(P))                                              # mean ± sd per size
display(A.comparison_table(P, pairs=[(base, n) for n in names])[0])     # each size vs 300d


,folds,minutes,balanced_accuracy,accuracy,macro_f1
system,,,,,
emb-ft-crawl-mean+func-promptcentred/logreg,50,0.5,0.4304 ± 0.0192,0.4305 ± 0.0192,0.4238 ± 0.0203
emb-ft-crawl-pca200-mean+func-promptcentred/logreg,50,0.3,0.3856 ± 0.0196,0.3857 ± 0.0196,0.3807 ± 0.0193
emb-ft-crawl-pca100-mean+func-promptcentred/logreg,50,0.2,0.3741 ± 0.0211,0.3742 ± 0.0210,0.3677 ± 0.0211
emb-ft-crawl-pca50-mean+func-promptcentred/logreg,50,0.3,0.3697 ± 0.0186,0.3698 ± 0.0186,0.3621 ± 0.0177


,A,B,folds,mean B−A,P(A>B),P(ROPE),P(B>A)
0,emb-ft-crawl-mean+func-promptcentred/logreg,emb-ft-crawl-pca50-mean+func-promptcentred/logreg,50,-0.0607,1.0000,0.0000,0.0
1,emb-ft-crawl-mean+func-promptcentred/logreg,emb-ft-crawl-pca100-mean+func-promptcentred/logreg,50,-0.0563,0.9999,0.0001,0.0
2,emb-ft-crawl-mean+func-promptcentred/logreg,emb-ft-crawl-pca200-mean+func-promptcentred/logreg,50,-0.0448,0.9990,0.0009,0.0


Now I try out different pooling approaches instead of mean. I want to try mean+max.

In [33]:
def pool_max(tokenizer, texts, keep=None):
    """Like WV.pool, but the per-dimension max over in-vocabulary tokens (OOV skipped)."""
    wv = tokenizer.wv
    out = np.zeros((len(texts), wv.dim), np.float32)
    for n, t in enumerate(texts):
        rows = sorted(wv.index[k] for s, k, _ in tokenizer.resolve(t) if k is not None and (keep is None or keep(s)))
        if rows:
            out[n] = np.asarray(wv.vectors[rows], np.float32).max(0)
    return out

Xmax = pool_max(tok, texts)
X3 = np.hstack([X2, Xmax])          # mean all | mean function words | max all  (900d)
r = H.run_cv(prompt_centre(X3, D.groups), D.y, D.groups, folds50, "logreg",
             name="emb-ft-crawl-mean+func+max-promptcentred/logreg")
H.save_result(r)
A.comparison_table({**R, two_lr["name"]: two_lr, r["name"]: r},
                   pairs=[(two_lr["name"], r["name"])])[0]

emb-ft-crawl-mean+func+max-promptcentred/logreg bal.acc 0.4166 ± 0.0188  (50 folds, 36s)


,A,B,folds,mean B−A,P(A>B),P(ROPE),P(B>A)
0,emb-ft-crawl-mean+func-promptcentred/logreg,emb-ft-crawl-mean+func+max-promptcentred/logreg,50,-0.0138,0.6382,0.3473,0.0145
